# World Model (Phase C) on Lite — Kaggle GPU execution

Runs the repo's `worldmodel` experiment on the free Kaggle GPU: the U-Net backbone plus a τ-day window of past states fused through a ConvLSTM at the encoder bottleneck (proposal Phase C). Training is teacher-forced; `predict()` rolls out autoregressively from the learned missing-history prior, and the notebook ends with the rollout **stability analysis** — error accumulation over a 2-year rollout vs teacher-forced 1-step.

Before running:
- **Accelerator → GPU** (right panel → Accelerator → GPU P100/T4)
- **Add a Secret** named `WANDB_API_KEY` (right panel → Secrets) so training curves + checkpoints land on wandb.ai (project `ClimX`, run `worldmodel-lite`). Without it, W&B falls back to offline mode.
- The lite data is already attached to the competition (`/kaggle/input/climx-main-track`).

## 1. Clone the repo + install dependencies (~2 min)

In [ ]:
import os, subprocess, sys, pathlib

# Idempotent: clone on first run, pull on re-runs (e.g. to pick up fixes).
repo = "/kaggle/working/climx"
if os.path.isdir(repo):
    r = subprocess.run(["git", "-C", repo, "pull", "--ff-only"], capture_output=True, text=True)
    print(r.stdout or r.stderr)
else:
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/nonewanit/climx-emulator.git", repo], check=True)
%cd /kaggle/working/climx
%pip install -q -e .
%pip install -q -r requirements-kaggle.txt

# Belt-and-braces: make the repo importable even if the editable install
# landed in a different environment (!pip vs kernel pip gotcha).
sys.path.insert(0, "/kaggle/working/climx/src")

# Auto-detect where the lite bundle actually lives (mount layout varies).
ncs = sorted(pathlib.Path("/kaggle/input").rglob("NorESM2-MM_r1i1p1f1_with_forcing_subsampled_16_*.nc"))
print(f"found {len(ncs)} lite files under /kaggle/input")
if not ncs:
    raise FileNotFoundError("No lite .nc files found — attach the climx-main-track competition data "
                            "to this notebook (right panel → Data sources).")
parents = {p.parent for p in ncs}
assert len(parents) == 1, f"lite files spread across dirs: {parents}"
os.environ["CLIMX_LITE_DIR"] = str(parents.pop())          # READ-ONLY mount
os.environ["CLIMX_PREPROCESS_DIR"] = "/kaggle/working/data/preprocessing_data_lite"
print("CLIMX_LITE_DIR =", os.environ["CLIMX_LITE_DIR"])

import climx_project
from climx_project import data, preprocess, paths
from climx_project.paths import ensure_dirs
ensure_dirs()

# W&B: load the API key from a notebook Secret (right panel → Secrets →
# add "WANDB_API_KEY"). Without it, training still runs — wandb falls
# back to OFFLINE mode (results/runs/wandb/); sync later with
# `!wandb sync results/runs/wandb`.
try:
    from kaggle_secrets import UserSecretsClient
    secret = UserSecretsClient().get_secret("WANDB_API_KEY")
    if secret:
        os.environ["WANDB_API_KEY"] = str(secret)
        print("WANDB_API_KEY loaded from notebook Secrets")
    else:
        print("WANDB_API_KEY secret not set — W&B will run offline")
except Exception as e:
    print("W&B secret unavailable:", e)

# Prove the dependency chain actually landed in this kernel:
import hydra, torch, xclim, lightning, wandb
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(), "| xclim", xclim.__version__)
print("wandb", wandb.__version__, "| key set:", bool(os.environ.get("WANDB_API_KEY")))

## 2. Locate the lite files in the bundle

In [ ]:
# The bundle layout may vary; the check prints what it needs.
try:
    data.check_lite_files()
    print("Lite files found under CLIMX_LITE_DIR")
except Exception as e:
    print("check failed:", e)
    print("--- searching for .nc files under /kaggle/input ---")
    import pathlib
    for p in pathlib.Path("/kaggle/input").rglob("*.nc"):
        print(p)
    raise

## 3. Preprocess (CPU, ~10-20 min — one-time)

In [ ]:
import time
t0 = time.time()
ds = data.open_datasets()
preprocess.run_preprocess_train(ds)
preprocess.run_preprocess_test(ds)
print(f"preprocess done in {(time.time()-t0)/60:.1f} min")

## 4. Train the world model (GPU, ~20-60 min)

Same data split and target space as the `unet` experiment (projections 2015-2080 train / 2081-2100 val, stationarized scaled anomalies), so the val-loss curves are directly comparable — the ablation ladder in the proposal. Watch it live on wandb.ai (project `ClimX`, run `worldmodel-lite`); checkpoints also appear there as artifacts because `log_model: all`.

In [ ]:
!python train.py experiment=worldmodel 2>&1 | tail -30
# smoke check first (optional): add  experiment.trainer.max_epochs=1 experiment.record=false
# shorter/longer memory:    experiment.model.model_params.tau=3
# force Lightning bars on:  experiment.trainer.enable_progress_bar=true
# scheduled sampling:       experiment.model.model_params.scheduled_sampling=0.25

## 4b. Checkpoints

Three kinds land in `results/models/worldmodel/`: the **best-val** checkpoint, **periodic epoch snapshots** (crash insurance), and the final `worldmodel.ckpt`. With the W&B key set they are ALSO uploaded as artifacts on the run page, so they survive kernel shutdown.

In [ ]:
from pathlib import Path
for p in sorted(Path("results/models/worldmodel").glob("*.ckpt")):
    print(f"{p.stat().st_size / 1e6:8.1f} MB  {p}")
# W&B copies: wandb.ai → project ClimX → run worldmodel-lite → Files → artifacts

## 5. Rollout stability analysis — Phase C's key experiment

Loads the best-val checkpoint, then compares **teacher-forced 1-step** predictions against a full **autoregressive rollout** (zero-init → the model's own predictions fed back) over 2 years of the SSP3-7.0 holdout. The gap between the curves is error accumulation; the first τ days show the spin-up transient from the learned missing-history prior. RMSE is in anomaly space, which equals raw-space daily RMSE (the reversal adds the same daily climatology to both sides).

In [ ]:
from pathlib import Path
import numpy as np, xarray as xr, torch, matplotlib.pyplot as plt

import climx_project
from climx_project import paths
from climx_project.models.worldmodel import WorldModelEmulator

# Best-val checkpoint from the training cell above.
ckpts = sorted(Path("results/models/worldmodel").glob("best-*.ckpt"))
assert ckpts, "no best-*.ckpt found — did the training cell run?"
best = str(ckpts[-1])
print("using", best)
em = WorldModelEmulator.from_checkpoint(best)
if torch.cuda.is_available():
    em.model = em.model.cuda()

# 2 years of the SSP370 holdout (2081-2082) in stationarized anomaly space.
X = xr.open_dataset(paths.X_TRAIN_NC)
Y = xr.open_dataset(paths.Y_TRAIN_NC)
Xh = X.sel(forcing_scenario="ssp370").sel(time=slice("2081-01-01", "2082-12-31"))
Yh = Y.sel(forcing_scenario="ssp370").sel(time=slice("2081-01-01", "2082-12-31"))
xa = Xh.to_array(dim="feature").transpose("feature", "time", "lat", "lon") \
        .values.astype(np.float32).transpose(1, 0, 2, 3)   # (T, F, H, W)
ya = Yh.to_array(dim="feature").transpose("feature", "time", "lat", "lon") \
        .values.astype(np.float32).transpose(1, 0, 2, 3)   # (T, 7, H, W)
print("profile window:", xa.shape, ya.shape)

# Warm start: the tau real days before 2081-01-01 (index 24090 in the
# 365-day noleap axis) — the SAME regime as actual inference. Without
# it the profile measures the obsolete zero-init spin-up.
tau = em.model.tau
init = Y.sel(forcing_scenario="ssp370").isel(time=slice(24090 - tau, 24090)) \
        .to_array(dim="feature").transpose("feature", "time", "lat", "lon") \
        .values.astype(np.float32).transpose(1, 0, 2, 3)
prof = em.rollout_profile(xa, ya, init_state=init)
d, one, ar = prof["days"], prof["one_step_rmse"], prof["rollout_rmse"]

plt.figure(figsize=(10, 4))
plt.plot(d, one, label="teacher-forced (1-step)", lw=2)
plt.plot(d, ar, label="autoregressive rollout", lw=2)
plt.axhline(np.nanmean(one), color="C0", ls=":", alpha=0.6)
plt.axhline(ar.mean(), color="C1", ls=":", alpha=0.6)
plt.xlabel("rollout day")
plt.ylabel("RMSE (anomaly space)")
plt.title("world model rollout stability — SSP370 2081-2082")
plt.legend(); plt.grid(alpha=0.3); plt.show()

tail_one = float(np.nanmean(one[-30:])); tail_ar = float(ar[-30:].mean())
print(f"1-step RMSE (last 30d): {tail_one:.4f}")
print(f"rollout RMSE (last 30d): {tail_ar:.4f}  (drift +{tail_ar - tail_one:.4f})")

# Log the profile to W&B (no-op offline without the key).
from climx_project import monitoring
run = monitoring.maybe_init_run("worldmodel-rollout-lite", project="ClimX",
                                config={"ckpt": best, "horizon_days": int(len(d))})
if run is not None:
    import wandb
    wandb.log({
        "rollout/one_step_rmse_tail30": tail_one,
        "rollout/ar_rmse_tail30": tail_ar,
        "rollout/drift_tail30": tail_ar - tail_one,
        "rollout/spinup_rmse_day7": float(ar[6]),
        "rollout/profile": wandb.Table(
            columns=["day", "one_step_rmse", "rollout_rmse"],
            data=[[int(t), None if np.isnan(o) else float(o), float(r)]
                  for t, o, r in zip(d, one, ar)]),
    })
    wandb.finish()
else:
    print("no W&B key — profile not logged; sync offline runs with "
          "!wandb sync results/runs/wandb")

## 6. Results

In [ ]:
from pathlib import Path
print(Path("results/scores.md").read_text())
# Leaderboard submission works directly from this notebook (Kaggle creds are
# built in):
#   !kaggle competitions submit -c climx-main-track -f results/submissions/submission_worldmodel.parquet -m "World model lite"